In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/bridge-node-data/nodes.csv
/kaggle/input/bridge-node-data/edges.csv


In [6]:
import networkx as nx
import pandas as pd
from itertools import product
import copy
import decimal


df = pd.read_csv("/kaggle/input/bridge-node-data/nodes.csv")
print("Before normalization:")
print("Clustering - Min:", df['clustering'].min(), "Max:", df['clustering'].max())
print("Indegree - Min:", df['indegree'].min(), "Max:", df['indegree'].max())
print("Eigencentrality - Min:", df['eigencentrality'].min(), "Max:", df['eigencentrality'].max())

# Apply min-max normalization to each column separately
df['clustering_normalized'] = (df['clustering'] - df['clustering'].min()) / (df['clustering'].max() - df['clustering'].min())
df['indegree_normalized'] = (df['indegree'] - df['indegree'].min()) / (df['indegree'].max() - df['indegree'].min())
df['eigencentrality_normalized'] = (df['eigencentrality'] - df['eigencentrality'].min()) / (df['eigencentrality'].max() - df['eigencentrality'].min())

# Read edge CSV file
edges = pd.read_csv('/kaggle/input/bridge-node-data/edges.csv')
nodes = df

# Create a NetworkX Graph
G = nx.from_pandas_edgelist(edges, source='Source', target='Target')

# Add node attributes from the node DataFrame
nx.set_node_attributes(G, df.set_index('Id')['indegree_normalized'].to_dict(), 'indegree')
nx.set_node_attributes(G, df.set_index('Id')['eigencentrality_normalized'].to_dict(), 'eigencentrality')
nx.set_node_attributes(G, df.set_index('Id')['clustering_normalized'].to_dict(), 'clustering')

# Calculate network-level metrics for the original graph
# original_avg_clustering = nx.average_clustering(G)
# original_avg_path_length = nx.average_shortest_path_length(G)
original_density = nx.density(G)
# print(f'avg clustering: {original_avg_clustering}')
# print(f'avg pathlength: {original_avg_path_length}')
print(f'orignal_density: {original_density}')

# Initialize a list to store the results
results = []
weight_df = []

# Iterate over all permutations of weights from 1 to 10
weight_combinations = product(range(1, 11), repeat=3)

for weights in weight_combinations:
    # Create a deep copy of the original graph
    G_copy = copy.deepcopy(G)

    indegree_weight, eigencentrality_weight, clustering_weight = weights

    # Calculate Bridge_Score for each node
    for node in G_copy.nodes():
        G_copy.nodes[node]['Bridge_Score'] = (indegree_weight * G_copy.nodes[node]['indegree'] +
                                               eigencentrality_weight * G_copy.nodes[node]['eigencentrality'] +
                                               clustering_weight * G_copy.nodes[node]['clustering'])

    # Identify top 10 nodes based on Bridge_Score
    top_10_nodes = sorted(G_copy.nodes(), key=lambda n: G_copy.nodes[n]['Bridge_Score'], reverse=True)[:10]
    print(top_10_nodes)
    
    print(weights)

    # Remove top 10 nodes from the graph
    weight_df.append((weights, top_10_nodes))
    G_copy.remove_nodes_from(top_10_nodes)
    print(G_copy)

    # Check if the graph is connected
    if nx.is_connected(G_copy):
        # Calculate network-level metrics for the modified graph
        # modified_avg_clustering = nx.average_clustering(G_copy)
        # modified_avg_path_length = nx.average_shortest_path_length(G_copy)
        modified_density = nx.density(G_copy)

        # Calculate the differences in network scores
        # clustering_diff = original_avg_clustering - modified_avg_clustering
        # path_length_diff = original_avg_path_length - modified_avg_path_length
        density_diff = original_density - modified_density

        # Add the results to the list
        results.append((weights, modified_density, density_diff))
    else:
        print("The graph is not connected for the weight combination:", indegree_weight, eigencentrality_weight, clustering_weight)

# Create a DataFrame from the results
results_df = pd.DataFrame(results, columns=['Weights', 'Density', 'Modified_Density'])
weight_df = pd.DataFrame(weight_df, columns=['Weights', 'top 10 bridge nodes'])

results_df.to_csv('results.csv', index=False)
weight_df.to_csv('weights.csv', index=False)
print("DataFrame saved to 'results.csv'")


Before normalization:
Clustering - Min: 0.0 Max: 0.5
Indegree - Min: 0 Max: 7
Eigencentrality - Min: 0.0 Max: 1.0
orignal_density: 0.0015849334393448593
['lomovkaa', 'warfakebelgorod', 'lu_di_z', 'voin_dv', 'Lunay14', 'orchestra_w', 'rustroyka1945', 'cyber_frontZ', 'novosti_efir', 'shkola_voenkora']
(1, 1, 1)
Graph with 1738 nodes and 2371 edges
['warfakebelgorod', 'lomovkaa', 'lu_di_z', 'shkola_voenkora', 'HersonEnot', 'UAVDEV', 'rezervsvo', 'rusvarg', 'VKhersone', 'TarasInform']
(1, 1, 2)
Graph with 1738 nodes and 2384 edges
['warfakebelgorod', 'lomovkaa', 'lu_di_z', 'shkola_voenkora', 'HersonEnot', 'UAVDEV', 'rezervsvo', 'rusvarg', 'VKhersone', 'TarasInform']
(1, 1, 3)
Graph with 1738 nodes and 2384 edges
['warfakebelgorod', 'shkola_voenkora', 'HersonEnot', 'UAVDEV', 'rezervsvo', 'rusvarg', 'VKhersone', 'lomovkaa', 'lu_di_z', 'TarasInform']
(1, 1, 4)
Graph with 1738 nodes and 2384 edges
['warfakebelgorod', 'shkola_voenkora', 'HersonEnot', 'UAVDEV', 'rezervsvo', 'rusvarg', 'VKhersone

In [ ]:
results

In [9]:
import networkx as nx
import pandas as pd
import copy


df = pd.read_csv("/kaggle/input/bridge-node-data/nodes.csv")
print("Before normalization:")
print("Clustering - Min:", df['clustering'].min(), "Max:", df['clustering'].max())
print("Indegree - Min:", df['indegree'].min(), "Max:", df['indegree'].max())
print("Eigencentrality - Min:", df['eigencentrality'].min(), "Max:", df['eigencentrality'].max())

# Apply min-max normalization to each column separately
df['clustering_normalized'] = (df['clustering'] - df['clustering'].min()) / (df['clustering'].max() - df['clustering'].min())
df['indegree_normalized'] = (df['indegree'] - df['indegree'].min()) / (df['indegree'].max() - df['indegree'].min())
df['eigencentrality_normalized'] = (df['eigencentrality'] - df['eigencentrality'].min()) / (df['eigencentrality'].max() - df['eigencentrality'].min())

# Read edge CSV file
edges = pd.read_csv('/kaggle/input/bridge-node-data/edges.csv')
nodes = df

# Create a NetworkX Graph
G = nx.from_pandas_edgelist(edges, source='Source', target='Target')

# Add node attributes from the node DataFrame
nx.set_node_attributes(G, df.set_index('Id')['indegree_normalized'].to_dict(), 'indegree')
nx.set_node_attributes(G, df.set_index('Id')['eigencentrality_normalized'].to_dict(), 'eigencentrality')
nx.set_node_attributes(G, df.set_index('Id')['clustering_normalized'].to_dict(), 'clustering')

# Calculate network-level metrics for the original graph
original_avg_clustering = nx.average_clustering(G)
original_avg_path_length = nx.average_shortest_path_length(G)
og_density = nx.density(G)
og_diameter = nx.diameter(G)
print(f'den: {og_density}, dia: {og_diameter}')
print(f'avg clustering: {original_avg_clustering}')
print(f'avg pathlength: {original_avg_path_length}')

# Initialize a list to store the results
results = []

# Manually specify the desired weights
indegree_weight = 5
eigencentrality_weight = 5
clustering_weight = 10

# Create a deep copy of the original graph
G_copy = copy.deepcopy(G)

# Calculate Bridge_Score for each node
for node in G_copy.nodes():
    G_copy.nodes[node]['Bridge_Score'] = (indegree_weight * G_copy.nodes[node]['indegree'] +
                                           eigencentrality_weight * G_copy.nodes[node]['eigencentrality'] +
                                           clustering_weight * G_copy.nodes[node]['clustering'])

# Identify top 10 nodes based on Bridge_Score
top_10_nodes = sorted(G_copy.nodes(), key=lambda n: G_copy.nodes[n]['Bridge_Score'], reverse=True)[:10]
print(top_10_nodes)

# Remove top 10 nodes from the graph
G_copy.remove_nodes_from(top_10_nodes)
print(G_copy)
print(nx.is_connected(G))
print(G.is_directed())
# Calculate network-level metrics for the modified graph
modified_avg_clustering = nx.average_clustering(G_copy)
modified_avg_path_length = nx.average_shortest_path_length(G_copy)
new_density = nx.density(G_copy)
new_diameter = nx.diameter(G_copy)
print(f'new_den: {new_density}, new_dia: {new_diameter}')
      
# Calculate the differences in network scores
clustering_diff = original_avg_clustering - modified_avg_clustering
path_length_diff = original_avg_path_length - modified_avg_path_length

# Add the results to the list
results.append((indegree_weight, eigencentrality_weight, clustering_weight, clustering_diff, path_length_diff, modified_avg_clustering, modified_avg_path_length))

# Create a DataFrame from the results
results_df = pd.DataFrame(results, columns=['Indegree_Weight', 'Eigencentrality_Weight', 'Clustering_Weight', 'Clustering_Diff', 'Path_Length_Diff',
                                            'Modified_Avg_Clustering', 'Modified_Avg_Path_Length'])



Before normalization:
Clustering - Min: 0.0 Max: 0.5
Indegree - Min: 0 Max: 7
Eigencentrality - Min: 0.0 Max: 1.0
den: 0.0015849334393448593, dia: 8
avg clustering: 0.08721388497664376
avg pathlength: 3.118987240630882
['warfakebelgorod', 'lomovkaa', 'lu_di_z', 'shkola_voenkora', 'HersonEnot', 'UAVDEV', 'rezervsvo', 'rusvarg', 'VKhersone', 'TarasInform']
Graph with 1738 nodes and 2384 edges
True
False
new_den: 0.00157938008006874, new_dia: 8


In [ ]:
results_df

In [ ]:
import networkx as nx
import pandas as pd
import copy

df = pd.read_csv("/kaggle/input/bridge-node-data/nodes.csv")
print("Before normalization:")
print("Clustering - Min:", df['clustering'].min(), "Max:", df['clustering'].max())
print("Indegree - Min:", df['indegree'].min(), "Max:", df['indegree'].max())
print("Eigencentrality - Min:", df['eigencentrality'].min(), "Max:", df['eigencentrality'].max())

# Apply min-max normalization to each column separately
df['clustering_normalized'] = (df['clustering'] - df['clustering'].min()) / (df['clustering'].max() - df['clustering'].min())
df['indegree_normalized'] = (df['indegree'] - df['indegree'].min()) / (df['indegree'].max() - df['indegree'].min())
df['eigencentrality_normalized'] = (df['eigencentrality'] - df['eigencentrality'].min()) / (df['eigencentrality'].max() - df['eigencentrality'].min())

# Read edge CSV file
edges = pd.read_csv('/kaggle/input/bridge-node-data/edges.csv')
nodes = df

# Create a NetworkX Graph
G = nx.from_pandas_edgelist(edges, source='Source', target='Target')

# Add node attributes from the node DataFrame
nx.set_node_attributes(G, df.set_index('Id')['indegree_normalized'].to_dict(), 'indegree')
nx.set_node_attributes(G, df.set_index('Id')['eigencentrality_normalized'].to_dict(), 'eigencentrality')
nx.set_node_attributes(G, df.set_index('Id')['clustering_normalized'].to_dict(), 'clustering')

# Calculate network-level metrics for the original graph
original_avg_clustering = nx.average_clustering(G)
original_avg_path_length = nx.average_shortest_path_length(G)
print(f'avg clustering: {original_avg_clustering}')
print(f'avg pathlength: {original_avg_path_length}')

# Initialize a list to store the results
results = []

# Manually specify the desired weights
# indegree_weight, eigencentrality_weight, clustering_weight = 3, 1, 2
# indegree_weight, eigencentrality_weight, clustering_weight = 9, 3, 6
# indegree_weight, eigencentrality_weight, clustering_weight = 6, 5, 4
# indegree_weight, eigencentrality_weight, clustering_weight = 6, 2, 4
# indegree_weight, eigencentrality_weight, clustering_weight = 6, 1, 4
indegree_weight, eigencentrality_weight, clustering_weight = 7, 5, 5
# indegree_weight, eigencentrality_weight, clustering_weight = 7, 6, 5
# indegree_weight, eigencentrality_weight, clustering_weight = 7, 7, 5
# indegree_weight, eigencentrality_weight, clustering_weight = 7, 8, 5
# indegree_weight, eigencentrality_weight, clustering_weight = 7, 9, 5
# indegree_weight, eigencentrality_weight, clustering_weight = 4, 8, 3
# indegree_weight, eigencentrality_weight, clustering_weight = 7, 10, 5
# indegree_weight, eigencentrality_weight, clustering_weight = 8, 1, 5
# indegree_weight, eigencentrality_weight, clustering_weight = 4, 7, 3
# indegree_weight, eigencentrality_weight, clustering_weight = 6, 3, 4
# indegree_weight, eigencentrality_weight, clustering_weight = 9, 2, 6
# indegree_weight, eigencentrality_weight, clustering_weight = 9, 1, 6
# indegree_weight, eigencentrality_weight, clustering_weight = 9, 4, 6
# indegree_weight, eigencentrality_weight, clustering_weight = 9, 5, 6


# Create a deep copy of the original graph
G_copy = copy.deepcopy(G)

# Calculate Bridge_Score for each node
for node in G_copy.nodes():
    G_copy.nodes[node]['Bridge_Score'] = (indegree_weight * G_copy.nodes[node]['indegree'] +
                                           eigencentrality_weight * G_copy.nodes[node]['eigencentrality'] +
                                           clustering_weight * G_copy.nodes[node]['clustering'])

# Identify top 10 nodes based on Bridge_Score
top_10_nodes = sorted(G_copy.nodes(), key=lambda n: G_copy.nodes[n]['Bridge_Score'], reverse=True)[:10]
print(top_10_nodes)

# Remove top 10 nodes from the graph
G_copy.remove_nodes_from(top_10_nodes)
print(G_copy)
print(nx.is_connected(G_copy))
print(G_copy.is_directed())
# Calculate network-level metrics for the modified graph
modified_avg_clustering = nx.average_clustering(G_copy)
modified_avg_path_length = nx.average_shortest_path_length(G_copy)

# Calculate the differences in network scores
clustering_diff = original_avg_clustering - modified_avg_clustering
path_length_diff = original_avg_path_length - modified_avg_path_length
combined_eval = clustering_diff - path_length_diff
# Add the results to the list
results.append((indegree_weight, eigencentrality_weight, clustering_weight, clustering_diff, path_length_diff, modified_avg_clustering, modified_avg_path_length, combined_eval))

# Create a DataFrame from the results
results_df = pd.DataFrame(results, columns=['Indegree_Weight', 'Eigencentrality_Weight', 'Clustering_Weight', 'Clustering_Diff', 'Path_Length_Diff',
                                            'Modified_Avg_Clustering', 'Modified_Avg_Path_Length', 'Combined_Evaluation'])

print(results_df)

In [ ]:
checking diffrence for top 10 vs top 10-20

In [ ]:
import networkx as nx
import pandas as pd
import copy

df = pd.read_csv("/kaggle/input/bridge-node-data/nodes.csv")
print("Before normalization:")
print("Clustering - Min:", df['clustering'].min(), "Max:", df['clustering'].max())
print("Indegree - Min:", df['indegree'].min(), "Max:", df['indegree'].max())
print("Eigencentrality - Min:", df['eigencentrality'].min(), "Max:", df['eigencentrality'].max())

# Apply min-max normalization to each column separately
df['clustering_normalized'] = (df['clustering'] - df['clustering'].min()) / (df['clustering'].max() - df['clustering'].min())
df['indegree_normalized'] = (df['indegree'] - df['indegree'].min()) / (df['indegree'].max() - df['indegree'].min())
df['eigencentrality_normalized'] = (df['eigencentrality'] - df['eigencentrality'].min()) / (df['eigencentrality'].max() - df['eigencentrality'].min())

# Read edge CSV file
edges = pd.read_csv('/kaggle/input/bridge-node-data/edges.csv')
nodes = df

# Create a NetworkX Graph
G = nx.from_pandas_edgelist(edges, source='Source', target='Target')

# Add node attributes from the node DataFrame
nx.set_node_attributes(G, df.set_index('Id')['indegree_normalized'].to_dict(), 'indegree')
nx.set_node_attributes(G, df.set_index('Id')['eigencentrality_normalized'].to_dict(), 'eigencentrality')
nx.set_node_attributes(G, df.set_index('Id')['clustering_normalized'].to_dict(), 'clustering')

# Calculate network-level metrics for the original graph
original_avg_clustering = nx.average_clustering(G)
original_avg_path_length = nx.average_shortest_path_length(G)
print(f'avg clustering: {original_avg_clustering}')
print(f'avg pathlength: {original_avg_path_length}')

# Initialize a list to store the results
results = []

# Manually specify the desired weights
indegree_weight, eigencentrality_weight, clustering_weight = 7,5,5

# Create deep copies of the original graph
G_copy_1 = copy.deepcopy(G)
G_copy_2 = copy.deepcopy(G)

# Calculate Bridge_Score for each node in each copy
for node in G_copy_1.nodes():
    G_copy_1.nodes[node]['Bridge_Score'] = (indegree_weight * G_copy_1.nodes[node]['indegree'] +
                                           eigencentrality_weight * G_copy_1.nodes[node]['eigencentrality'] +
                                           clustering_weight * G_copy_1.nodes[node]['clustering'])
    
for node in G_copy_2.nodes():
    G_copy_2.nodes[node]['Bridge_Score'] = (indegree_weight * G_copy_2.nodes[node]['indegree'] +
                                           eigencentrality_weight * G_copy_2.nodes[node]['eigencentrality'] +
                                           clustering_weight * G_copy_2.nodes[node]['clustering'])

# Identify top 10 nodes based on Bridge_Score in each copy
top_10_nodes_1 = sorted(G_copy_1.nodes(), key=lambda n: G_copy_1.nodes[n]['Bridge_Score'], reverse=True)[:5]
print(top_10_nodes_1)
top_10_nodes_2 = sorted(G_copy_2.nodes(), key=lambda n: G_copy_2.nodes[n]['Bridge_Score'], reverse=True)[5:10]
print(top_10_nodes_2)

# Remove top 10 nodes from each copy
G_copy_1.remove_nodes_from(top_10_nodes_1)
G_copy_2.remove_nodes_from(top_10_nodes_2)

# Calculate network-level metrics for each modified graph
modified_avg_clustering_1 = nx.average_clustering(G_copy_1)
modified_avg_path_length_1 = nx.average_shortest_path_length(G_copy_1)

modified_avg_clustering_2 = nx.average_clustering(G_copy_2)
modified_avg_path_length_2 = nx.average_shortest_path_length(G_copy_2)

# Calculate the differences in network scores for each modified graph
clustering_diff_1 = original_avg_clustering - modified_avg_clustering_1
path_length_diff_1 = original_avg_path_length - modified_avg_path_length_1

clustering_diff_2 = original_avg_clustering - modified_avg_clustering_2
path_length_diff_2 = original_avg_path_length - modified_avg_path_length_2

# Calculate the combined evaluation metric for each modified graph
combined_eval_1 = clustering_diff_1 - path_length_diff_1
combined_eval_2 = clustering_diff_2 - path_length_diff_2

# Add the results to the list
results.append((indegree_weight, eigencentrality_weight, clustering_weight, clustering_diff_1, path_length_diff_1, modified_avg_clustering_1, modified_avg_path_length_1, combined_eval_1))
results.append((indegree_weight, eigencentrality_weight, clustering_weight, clustering_diff_2, path_length_diff_2, modified_avg_clustering_2, modified_avg_path_length_2, combined_eval_2))

# Create a DataFrame from the results
results_df = pd.DataFrame(results, columns=['Indegree_Weight', 'Eigencentrality_Weight', 'Clustering_Weight', 
                                            'Clustering_Diff', 'Path_Length_Diff', 'Modified_Avg_Clustering', 
                                            'Modified_Avg_Path_Length', 'Combined_Evaluation'])

# Print the results
print(results_df)


In [ ]:
import networkx as nx
import pandas as pd
import copy

df = pd.read_csv("/kaggle/input/bridge-node-data/nodes.csv")
print("Before normalization:")
print("Clustering - Min:", df['clustering'].min(), "Max:", df['clustering'].max())
print("Indegree - Min:", df['indegree'].min(), "Max:", df['indegree'].max())
print("Eigencentrality - Min:", df['eigencentrality'].min(), "Max:", df['eigencentrality'].max())

# Apply min-max normalization to each column separately
df['clustering_normalized'] = (df['clustering'] - df['clustering'].min()) / (df['clustering'].max() - df['clustering'].min())
df['indegree_normalized'] = (df['indegree'] - df['indegree'].min()) / (df['indegree'].max() - df['indegree'].min())
df['eigencentrality_normalized'] = (df['eigencentrality'] - df['eigencentrality'].min()) / (df['eigencentrality'].max() - df['eigencentrality'].min())

# Read edge CSV file
edges = pd.read_csv('/kaggle/input/bridge-node-data/edges.csv')
nodes = df

# Create a NetworkX Graph
G = nx.from_pandas_edgelist(edges, source='Source', target='Target')

# Add node attributes from the node DataFrame
nx.set_node_attributes(G, df.set_index('Id')['indegree_normalized'].to_dict(), 'indegree')
nx.set_node_attributes(G, df.set_index('Id')['eigencentrality_normalized'].to_dict(), 'eigencentrality')
nx.set_node_attributes(G, df.set_index('Id')['clustering_normalized'].to_dict(), 'clustering')

# Calculate network-level metrics for the original graph
original_avg_clustering = nx.average_clustering(G)
original_avg_path_length = nx.average_shortest_path_length(G)
print(f'Original Avg Clustering: {original_avg_clustering}')
print(f'Original Avg Path Length: {original_avg_path_length}')

# Initialize a list to store the results
results = []

# Specify the desired weight combinations
weight_combinations = [
    (3, 1, 2), (9, 3, 6), (6, 5, 4), (6, 2, 4), (6, 1, 4), (7, 5, 5), (7, 6, 5), (7, 7, 5),
    (7, 8, 5), (7, 9, 5), (4, 8, 3), (7, 10, 5), (8, 1, 5), (4, 7, 3), (6, 3, 4), (9, 2, 6),
    (9, 1, 6), (9, 4, 6), (9, 5, 6)
]

# Iterate through each weight combination
for idx, (indegree_weight, eigencentrality_weight, clustering_weight) in enumerate(weight_combinations, start=1):
    print(f"\nWeight Combination {idx}: Indegree Weight: {indegree_weight}, Eigencentrality Weight: {eigencentrality_weight}, Clustering Weight: {clustering_weight}")
    
    # Create deep copies of the original graph
    G_copy_1 = copy.deepcopy(G)
    G_copy_2 = copy.deepcopy(G)

    # Calculate Bridge_Score for each node in each copy
    for node in G_copy_1.nodes():
        G_copy_1.nodes[node]['Bridge_Score'] = (indegree_weight * G_copy_1.nodes[node]['indegree'] +
                                                eigencentrality_weight * G_copy_1.nodes[node]['eigencentrality'] +
                                                clustering_weight * G_copy_1.nodes[node]['clustering'])
    for node in G_copy_2.nodes():
        G_copy_2.nodes[node]['Bridge_Score'] = (indegree_weight * G_copy_2.nodes[node]['indegree'] +
                                                eigencentrality_weight * G_copy_2.nodes[node]['eigencentrality'] +
                                                clustering_weight * G_copy_2.nodes[node]['clustering'])

    # Identify top 10 nodes based on Bridge_Score in each copy
    top_10_nodes_1 = sorted(G_copy_1.nodes(), key=lambda n: G_copy_1.nodes[n]['Bridge_Score'], reverse=True)[:5]
    print("Top 10 Nodes (Copy 1):", top_10_nodes_1)
    top_10_nodes_2 = sorted(G_copy_2.nodes(), key=lambda n: G_copy_2.nodes[n]['Bridge_Score'], reverse=True)[5:10]
    print("Top 10 Nodes (Copy 2):", top_10_nodes_2)



In [11]:
import networkx as nx
import pandas as pd
import copy

df = pd.read_csv("/kaggle/input/bridge-node-data/nodes.csv")
print("Before normalization:")
print("Clustering - Min:", df['clustering'].min(), "Max:", df['clustering'].max())
print("Indegree - Min:", df['indegree'].min(), "Max:", df['indegree'].max())
print("Eigencentrality - Min:", df['eigencentrality'].min(), "Max:", df['eigencentrality'].max())

# Apply min-max normalization to each column separately
df['clustering_normalized'] = (df['clustering'] - df['clustering'].min()) / (df['clustering'].max() - df['clustering'].min())
df['indegree_normalized'] = (df['indegree'] - df['indegree'].min()) / (df['indegree'].max() - df['indegree'].min())
df['eigencentrality_normalized'] = (df['eigencentrality'] - df['eigencentrality'].min()) / (df['eigencentrality'].max() - df['eigencentrality'].min())

# Read edge CSV file
edges = pd.read_csv('/kaggle/input/bridge-node-data/edges.csv')
nodes = df

# Create a NetworkX Graph
G = nx.from_pandas_edgelist(edges, source='Source', target='Target')

# Add node attributes from the node DataFrame
nx.set_node_attributes(G, df.set_index('Id')['indegree_normalized'].to_dict(), 'indegree')
nx.set_node_attributes(G, df.set_index('Id')['eigencentrality_normalized'].to_dict(), 'eigencentrality')
nx.set_node_attributes(G, df.set_index('Id')['clustering_normalized'].to_dict(), 'clustering')

# Calculate network-level metrics for the original graph
original_avg_clustering = nx.average_clustering(G)
original_avg_path_length = nx.average_shortest_path_length(G)
print(f'Original Avg Clustering: {original_avg_clustering}')
print(f'Original Avg Path Length: {original_avg_path_length}')

# Initialize a list to store the results
results = []

# Specify the desired weight combinations
weight_combinations = [
    (3, 2, 2), (10, 4, 7), (9, 7, 6), (9, 6, 6), (9, 5, 6), (9, 4, 6), (4, 7, 3), (9, 3, 6), (9, 2, 6), (9, 1, 6), (4, 8, 3), (8, 1, 5), (3, 1, 2), (7, 10, 5), (7, 9, 5), (7, 8, 5), (7, 7, 5), (7, 6, 5), (7, 5, 5), (6, 5, 4), (6, 4, 4), (6, 3, 4), (6, 2, 4), (10, 3, 7), (6, 1, 4), (10, 7, 7), (10, 9, 7), (10, 5, 7), (10, 10, 7), (10, 6, 7), (10, 8, 7)
]


# Iterate through each weight combination
for idx, (indegree_weight, eigencentrality_weight, clustering_weight) in enumerate(weight_combinations, start=1):
    print(f"\nWeight Combination {idx}: Indegree Weight: {indegree_weight}, Eigencentrality Weight: {eigencentrality_weight}, Clustering Weight: {clustering_weight}")
    
    # Create deep copies of the original graph
    G_copy_1 = copy.deepcopy(G)
    G_copy_2 = copy.deepcopy(G)

    # Calculate Bridge_Score for each node in each copy
    for node in G_copy_1.nodes():
        G_copy_1.nodes[node]['Bridge_Score'] = (indegree_weight * G_copy_1.nodes[node]['indegree'] +
                                                eigencentrality_weight * G_copy_1.nodes[node]['eigencentrality'] +
                                                clustering_weight * G_copy_1.nodes[node]['clustering'])
#     for node in G_copy_2.nodes():
#         G_copy_2.nodes[node]['Bridge_Score'] = (indegree_weight * G_copy_2.nodes[node]['indegree'] +
#                                                 eigencentrality_weight * G_copy_2.nodes[node]['eigencentrality'] +
#                                                 clustering_weight * G_copy_2.nodes[node]['clustering'])

    # Identify top 10 nodes based on Bridge_Score in each copy
    top_10_nodes_1 = sorted(G_copy_1.nodes(), key=lambda n: G_copy_1.nodes[n]['Bridge_Score'], reverse=True)[:10]
    print("Top 10 Nodes (Copy 1):", top_10_nodes_1)
#     top_10_nodes_2 = sorted(G_copy_2.nodes(), key=lambda n: G_copy_2.nodes[n]['Bridge_Score'], reverse=True)[5:10]
#     print("Top 10 Nodes (Copy 2):", top_10_nodes_2)



Before normalization:
Clustering - Min: 0.0 Max: 0.5
Indegree - Min: 0 Max: 7
Eigencentrality - Min: 0.0 Max: 1.0
Original Avg Clustering: 0.08721388497664376
Original Avg Path Length: 3.118987240630882

Weight Combination 1: Indegree Weight: 3, Eigencentrality Weight: 2, Clustering Weight: 2
Top 10 Nodes (Copy 1): ['lomovkaa', 'lu_di_z', 'warfakebelgorod', 'zarussia_1', 'ua_tribunal', 'zachitniki', 'voin_dv', 'Lunay14', 'orchestra_w', 'rustroyka1945']

Weight Combination 2: Indegree Weight: 10, Eigencentrality Weight: 4, Clustering Weight: 7
Top 10 Nodes (Copy 1): ['lomovkaa', 'lu_di_z', 'warfakebelgorod', 'zarussia_1', 'ua_tribunal', 'zachitniki', 'voin_dv', 'Lunay14', 'orchestra_w', 'rustroyka1945']

Weight Combination 3: Indegree Weight: 9, Eigencentrality Weight: 7, Clustering Weight: 6
Top 10 Nodes (Copy 1): ['lomovkaa', 'lu_di_z', 'warfakebelgorod', 'zarussia_1', 'ua_tribunal', 'zachitniki', 'voin_dv', 'Lunay14', 'orchestra_w', 'rustroyka1945']

Weight Combination 4: Indegree We

In [13]:
import networkx as nx
import pandas as pd
import copy

df = pd.read_csv("/kaggle/input/bridge-node-data/nodes.csv")
print("Before normalization:")
print("Clustering - Min:", df['clustering'].min(), "Max:", df['clustering'].max())
print("Indegree - Min:", df['indegree'].min(), "Max:", df['indegree'].max())
print("Eigencentrality - Min:", df['eigencentrality'].min(), "Max:", df['eigencentrality'].max())

# Apply min-max normalization to each column separately
df['clustering_normalized'] = (df['clustering'] - df['clustering'].min()) / (df['clustering'].max() - df['clustering'].min())
df['indegree_normalized'] = (df['indegree'] - df['indegree'].min()) / (df['indegree'].max() - df['indegree'].min())
df['eigencentrality_normalized'] = (df['eigencentrality'] - df['eigencentrality'].min()) / (df['eigencentrality'].max() - df['eigencentrality'].min())

# Read edge CSV file
edges = pd.read_csv('/kaggle/input/bridge-node-data/edges.csv')
nodes = df

# Create a NetworkX Graph
G = nx.from_pandas_edgelist(edges, source='Source', target='Target')

# Add node attributes from the node DataFrame
nx.set_node_attributes(G, df.set_index('Id')['indegree_normalized'].to_dict(), 'indegree')
nx.set_node_attributes(G, df.set_index('Id')['eigencentrality_normalized'].to_dict(), 'eigencentrality')
nx.set_node_attributes(G, df.set_index('Id')['clustering_normalized'].to_dict(), 'clustering')

# Calculate network-level metrics for the original graph
original_avg_clustering = nx.average_clustering(G)
original_avg_path_length = nx.average_shortest_path_length(G)
original_density = nx.density(G)
original_transitivity = nx.transitivity(G)

print(f'Original Avg Clustering: {original_avg_clustering}')
print(f'Original Avg Path Length: {original_avg_path_length}')
print(f'Original Density: {original_density}')
print(f'Original Transitivity: {original_transitivity}')

# Initialize a list to store the results
results = []

# Specify the desired weight combinations
weight_combinations = [
    (3, 2, 2), (10, 4, 7), (9, 7, 6), (9, 6, 6), (9, 5, 6), (9, 4, 6), (4, 7, 3), (9, 3, 6), (9, 2, 6), (9, 1, 6),
    (4, 8, 3), (8, 1, 5), (3, 1, 2), (7, 10, 5), (7, 9, 5), (7, 8, 5), (7, 7, 5), (7, 6, 5), (7, 5, 5), (6, 5, 4),
    (6, 4, 4), (6, 3, 4), (6, 2, 4), (10, 3, 7), (6, 1, 4), (10, 7, 7), (10, 9, 7), (10, 5, 7), (10, 10, 7), (10, 6, 7), (10, 8, 7)
]

# Iterate through each weight combination
for idx, (indegree_weight, eigencentrality_weight, clustering_weight) in enumerate(weight_combinations, start=1):
    print(f"\nWeight Combination {idx}: Indegree Weight: {indegree_weight}, Eigencentrality Weight: {eigencentrality_weight}, Clustering Weight: {clustering_weight}")

    # Create a deep copy of the original graph
    G_copy = copy.deepcopy(G)

    # Calculate Bridge_Score for each node
    for node in G_copy.nodes():
        G_copy.nodes[node]['Bridge_Score'] = (indegree_weight * G_copy.nodes[node]['indegree'] +
                                              eigencentrality_weight * G_copy.nodes[node]['eigencentrality'] +
                                              clustering_weight * G_copy.nodes[node]['clustering'])

    # Identify top 10 nodes based on Bridge_Score
    top_10_nodes = sorted(G_copy.nodes(), key=lambda n: G_copy.nodes[n]['Bridge_Score'], reverse=True)[:10]
    print("Top 10 Nodes:", top_10_nodes)

    # Remove top 10 nodes from the graph
    G_copy.remove_nodes_from(top_10_nodes)

    # Check if the graph is connected
    if nx.is_connected(G_copy):
        print("Graph is connected")

        # Calculate network-level metrics for the modified graph
        modified_avg_clustering = nx.average_clustering(G_copy)
        modified_avg_path_length = nx.average_shortest_path_length(G_copy)
        modified_density = nx.density(G_copy)
        modified_transitivity = nx.transitivity(G_copy)

        # Calculate the differences in network scores
        clustering_diff = original_avg_clustering - modified_avg_clustering 
        path_length_diff = original_avg_path_length - modified_avg_path_length 
        density_diff =  original_density - modified_density 
        transitivity_diff =  original_transitivity - modified_density 

        # Store the results
        results.append((indegree_weight, eigencentrality_weight, clustering_weight,
                        clustering_diff, path_length_diff, density_diff, transitivity_diff))

        print(f"Clustering Difference: {clustering_diff}")
        print(f"Path Length Difference: {path_length_diff}")
        print(f"Density Difference: {density_diff}")
        print(f"Transitivity Difference: {transitivity_diff}")
    else:
        print("Graph is not connected")

# Create a DataFrame from the results
results_df = pd.DataFrame(results, columns=['Indegree Weight', 'Eigencentrality Weight', 'Clustering Weight',
                                            'Clustering Difference', 'Path Length Difference', 'Density Difference',
                                            'Transitivity Difference'])

# Identify the weight combination with the largest clustering difference
max_clustering_diff_row = results_df['Clustering Difference'].idxmax()
max_clustering_diff_weights = results_df.loc[max_clustering_diff_row, ['Indegree Weight', 'Eigencentrality Weight', 'Clustering Weight']].values
print(f"\nWeight combination with the largest clustering difference: {max_clustering_diff_weights}")

# Identify the weight combination with the smallest path length difference
min_path_length_diff_row = results_df['Path Length Difference'].idxmin()
min_path_length_diff_weights = results_df.loc[min_path_length_diff_row, ['Indegree Weight', 'Eigencentrality Weight', 'Clustering Weight']].values
print(f"\nWeight combination with the smallest path length difference: {min_path_length_diff_weights}")

# Identify the weight combination with the largest density difference
max_density_diff_row = results_df['Density Difference'].idxmax()
max_density_diff_weights = results_df.loc[max_density_diff_row, ['Indegree Weight', 'Eigencentrality Weight', 'Clustering Weight']].values
print(f"\nWeight combination with the largest density difference: {max_density_diff_weights}")

# Identify the weight combination with the largest transitivity difference
max_transitivity_diff_row = results_df['Transitivity Difference'].idxmax()
max_transitivity_diff_weights = results_df.loc[max_transitivity_diff_row, ['Indegree Weight', 'Eigencentrality Weight', 'Clustering Weight']].values
print(f"\nWeight combination with the largest transitivity difference: {max_transitivity_diff_weights}")



Before normalization:
Clustering - Min: 0.0 Max: 0.5
Indegree - Min: 0 Max: 7
Eigencentrality - Min: 0.0 Max: 1.0
Original Avg Clustering: 0.08721388497664376
Original Avg Path Length: 3.118987240630882
Original Density: 0.0015849334393448593
Original Transitivity: 0.0024526587939347356

Weight Combination 1: Indegree Weight: 3, Eigencentrality Weight: 2, Clustering Weight: 2
Top 10 Nodes: ['lomovkaa', 'lu_di_z', 'warfakebelgorod', 'zarussia_1', 'ua_tribunal', 'zachitniki', 'voin_dv', 'Lunay14', 'orchestra_w', 'rustroyka1945']
Graph is connected
Clustering Difference: 0.003169365190309026
Path Length Difference: -0.010506026135290725
Density Difference: 1.8140700518277757e-05
Transitivity Difference: 0.0008858660551081541

Weight Combination 2: Indegree Weight: 10, Eigencentrality Weight: 4, Clustering Weight: 7
Top 10 Nodes: ['lomovkaa', 'lu_di_z', 'warfakebelgorod', 'zarussia_1', 'ua_tribunal', 'zachitniki', 'voin_dv', 'Lunay14', 'orchestra_w', 'rustroyka1945']
Graph is connected
Clu

In [ ]:
['lomovkaa', 'lu_di_z', 'warfakebelgorod', 'zarussia_1', 'ua_tribunal', 'zachitniki', 'voin_dv', 'Lunay14', 'orchestra_w', 'rustroyka1945']